# 2. Interventions triggered by prevalence

This notebook answers the STRIDE question directly. It explains how the threshold trigger is
built and then runs four intervention scenarios against the baseline from
[notebook 1](01-baseline-model.ipynb).

**The short version.** summer4 evaluates every flow rate from the model's *current* state at
every solver step. A rate can therefore be a function of TB prevalence, just as a force of
infection is a function of the number infectious. A trigger needs three pieces:

1. **a signal**: prevalence per 100,000, read from the compartments;
2. **a switch**: a function that is 0 below the threshold and 1 above it;
3. **memory**: so that the programme stays on once triggered, even after it has pushed
   prevalence back below the threshold. That memory is a pair of bookkeeping compartments.

None of this needs event handling, restarting the solver, or a second run. The whole thing is
one ODE, so it works with any solver, under `jax.jit`, `jax.vmap` and `jax.grad`, and in
calibration.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

import jax
import plotly.express as px
from summer4 import Param, Time

import tb_trigger.model as tbm

cm = tbm.build_model().compile()
threshold = tbm.DEFAULT_PARAMS["threshold"]

## 1. The signal: prevalence as a rate expression

`Reduce(sum_over=pop, where=selector)` sums the current compartment sizes that match the
selector. Prevalence per 100,000 is one `Reduce` divided by another:

```python
def prevalence_per_100k():
    return 1e5 * Reduce(sum_over=POP, where=people("I")) / Reduce(sum_over=POP, where=PEOPLE)
```

This is an *expression*, not a number. It describes a calculation that summer4 runs inside the
vector field at every step. In summer2 terms it does the job of a function flow or a computed
value, without leaving the compiled model.

In [ ]:
tbm.prevalence_per_100k()

## 2. The switch: a smooth step, wrapped with `defer`

`defer(fn)(arg, ...)` puts an ordinary Python function into a rate expression. Its arguments
can be expressions such as the prevalence above, `Param`s or `Time()`, and `fn` receives their
current values. It is summer2's `defer` from computegraph, essentially unchanged. The switch is
a logistic step:

```python
def _smooth_step(signal, threshold, width):
    on = jax.nn.sigmoid((signal.data - threshold) / width)   # 0 below, 1 above
    return GroupedRate(on, signal.properties)

switch_on = defer(_smooth_step)(prevalence_per_100k(), Param("threshold"), Param("threshold_width"))
```

(The `.data` / `GroupedRate` wrapping is there because `Reduce` returns a value grouped by
`pop`. That is a sharp edge in summer4 `v0.2.0a5`.)

**Why smooth and not `prevalence > threshold`?** A hard step works with a fixed-step Euler
solver. A smooth step with a narrow `threshold_width` (0.1 per 100,000 by default) has two
advantages. Adaptive solvers integrate across it efficiently, and model outputs stay
differentiable with respect to the threshold, which notebook 3 uses.

The figure evaluates the **actual model's** `trigger` flow, using `cm.observe` at states with
different prevalence. It shows the rate at which the programme leaves `waiting`, for three
switch widths. Away from the threshold the rate is essentially zero (below) or
`trigger_speed` = 365 per year, a mean of about one day to decide (above).

In [ ]:
y0 = np.asarray(cm.initial_state(tbm.DEFAULT_PARAMS).data).copy()
i_idx = cm.pmap.select(tbm.people("I"))
s_idx = cm.pmap.select(tbm.people("S"))


def trigger_rate(prevalence, width):
    y = y0.copy()
    y[i_idx] = prevalence
    y[s_idx] = 1e5 - prevalence
    params = tbm.scenario_params("baseline", threshold_width=width)
    return float(np.asarray(cm.observe(2000.0, y, params).flows["trigger"]).sum())


grid = np.linspace(340.0, 360.0, 201)
switch_curves = pd.DataFrame(
    {f"width {w}": [trigger_rate(p, w) for p in grid] for w in (0.1, 1.0, 3.0)}, index=grid
)
switch_curves.index.name = "prevalence per 100,000"
switch_curves.plot(
    title="Trigger flow rate as a function of prevalence",
    labels={"value": "rate leaving 'waiting' (per year)", "variable": "threshold_width"},
)

## 3. Memory: the programme compartments

If the switch multiplied the intervention directly, the intervention would be on only while
prevalence is above the threshold. That is a legitimate policy (the *reactive* scenario below),
but it is not "once prevalence reaches X, start the programme". That policy has to remember
that the threshold was reached.

An ODE remembers things through its state, so the model carries a tiny side model: one unit
of mass that moves through three compartments.

```
            trigger_speed × switch_on            1 / rollout_years
 waiting  ─────────────────────────────▶  scaling_up  ─────────────────▶  active
    ▲                                                                       │
    └────────────────────  stand_down × (1 − switch_on)  ───────────────────┘
```

* **`trigger`** moves mass out of `waiting` only while prevalence is above the threshold. Once
  the mass has left it does not come back, so the switch **latches**.
* **`roll_out`** gives a gradual scale-up with mean `rollout_years`. Coverage rises as
  `1 − exp(−t / rollout_years)`.
* **`stand_down`** is zero in the latched scenarios. In the reactive scenario it returns mass
  to `waiting` whenever prevalence falls below the threshold.

The intervention flows are multiplied by the mass in `active`, which is 0 before the trigger
and 1 at full coverage. The programme compartments are excluded from every population count,
so they cannot leak into the epidemiology.

## 4. The scenarios

Every scenario is the same compiled model with different parameter values. `SCENARIOS` holds
only the changes from `DEFAULT_PARAMS`:

| Scenario | What happens when prevalence reaches 350 per 100,000 |
|---|---|
| `baseline` | Nothing (intervention strengths are zero) |
| `acf_triggered` | Active case finding (half the population screened each year, sensitivity 70%) rolls out with a mean of 3 months and stays on |
| `acf_triggered_slow_rollout` | The same ACF, rolled out with a mean of 3 years |
| `acf_reactive` | ACF runs only while prevalence is above the threshold (stands down below it) |
| `acf_tpt_triggered` | ACF plus preventive therapy for the latently infected, rolled out in 3 months, stays on |

In [ ]:
pd.DataFrame(tbm.SCENARIOS).T.fillna("")

In [ ]:
frames = {name: tbm.indicators(tbm.run(cm, tbm.scenario_params(name))) for name in tbm.SCENARIOS}


def long(column, start=2015.0):
    """One indicator for every scenario, in long format for plotting."""
    return (
        pd.DataFrame({tbm.SCENARIO_LABELS[n]: f.loc[start:, column] for n, f in frames.items()})
        .rename_axis("year")
        .reset_index()
        .melt(id_vars="year", var_name="scenario", value_name=column)
    )

### Prevalence

Before about 2030 every curve is the baseline, because nothing has been triggered. When
baseline prevalence reaches the dashed threshold the scenarios separate:

* **ACF on trigger** (fast roll-out) pulls prevalence down within a couple of years and holds it
  around 240 per 100,000, even though transmission keeps rising until 2035.
* **3-year roll-out** overshoots only a little (peak about 355), but it comes down slowly. In
  2035 it is still near 300, against 250 with the fast roll-out, and it reaches the same level
  only by the mid-2040s.
* **Reactive ACF** holds prevalence *at* the threshold. See section 6.
* **ACF + TPT** keeps falling. Preventive therapy drains the latent pool that feeds future
  cases.

In [ ]:
fig = long("prevalence").plot(
    x="year", y="prevalence", color="scenario",
    title="Active TB prevalence by scenario",
    labels={"prevalence": "per 100,000"},
)
fig.add_hline(y=threshold, line_dash="dash", annotation_text="threshold")
fig

### Programme coverage

The fraction of the programme in `active`. The three latched scenarios switch on at the same
moment, the baseline crossing year, and stay on. The slow roll-out follows the exponential
`1 − exp(−t/3)`. The reactive programme never reaches full coverage. It runs at just the
intensity needed to stop prevalence rising.

In [ ]:
fig = long("programme_active").plot(
    x="year", y="programme_active", color="scenario",
    title="Programme coverage (fraction active)",
    labels={"programme_active": "coverage"},
)
fig.add_vline(x=tbm.crossing_year(frames["baseline"]["prevalence"], threshold), line_dash="dash", annotation_text="baseline crosses threshold")
fig

### Zooming in on the trigger

The two years around the trigger in the `acf_triggered` scenario, with prevalence on the left
axis and the three programme compartments on the right. Prevalence touches 350, the programme
mass leaves `waiting` within days, spends a few months `scaling_up`, and prevalence turns
over as coverage builds.

In [ ]:
from plotly.subplots import make_subplots

zoom = frames["acf_triggered"].loc[2029.0:2032.0]
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=zoom.index, y=zoom["prevalence"], name="prevalence per 100,000")
for col in ("programme_waiting", "programme_scaling_up", "programme_active"):
    fig.add_scatter(x=zoom.index, y=zoom[col], name=col.replace("programme_", ""), line_dash="dot", secondary_y=True)
fig.add_hline(y=threshold, line_dash="dash")
fig.update_layout(title="The trigger moment, ACF on trigger", xaxis_title="year")
fig.update_yaxes(title_text="prevalence per 100,000", secondary_y=False)
fig.update_yaxes(title_text="fraction of programme", secondary_y=True)
fig

### Incidence, notifications and deaths

Notifications show the familiar active case finding bump. Starting ACF finds the backlog of
prevalent cases, so notifications jump above baseline and then settle lower as prevalence
falls. Incidence and deaths fall below baseline for every intervention. With ACF + TPT,
incidence falls furthest, because TPT acts on the latent pool that progresses to disease.

In [ ]:
indicators_long = pd.concat(
    [long(c).rename(columns={c: "value"}).assign(indicator=c) for c in ("incidence", "notifications", "tb_deaths")]
)
fig = indicators_long.plot(
    x="year", y="value", color="scenario", facet_row="indicator", height=800,
    title="Incidence, notifications and TB deaths (per 100,000 per year)",
)
fig.update_yaxes(matches=None)
fig

### What the scenarios buy, 2025 to 2050

Cumulative incident cases and TB deaths averted relative to the baseline, per 100,000
population. The cumulative totals integrate the saved annual rates with the trapezoid rule. The ranking is ACF + TPT, then fast ACF, then slow roll-out, then reactive. A
programme that stands down as soon as prevalence dips back under the threshold averts the
least.

In [ ]:
def cumulative(frame, column, start=2025.0, end=2050.0):
    s = frame.loc[start:end, column]
    return float(np.trapezoid(s.to_numpy(), s.index.to_numpy()))


averted = pd.DataFrame(
    {
        tbm.SCENARIO_LABELS[n]: {
            "incident cases averted": cumulative(frames["baseline"], "incidence") - cumulative(f, "incidence"),
            "TB deaths averted": cumulative(frames["baseline"], "tb_deaths") - cumulative(f, "tb_deaths"),
        }
        for n, f in frames.items()
        if n != "baseline"
    }
).T
fig = averted.plot.bar(
    barmode="group", title="Averted 2025–2050, per 100,000 population",
    labels={"index": "scenario", "value": "per 100,000", "variable": ""},
)
fig

In [ ]:
averted.round(0)

## 5. Is the trigger firing at the right time?

Two checks.

**When does it fire?** Before the trigger, every scenario *is* the baseline. So the programme
should leave `waiting` exactly when baseline prevalence crosses 350. The table compares the
year baseline prevalence crosses the threshold with the year half of each scenario's programme
mass has left `waiting`. They agree to within a day.

In [ ]:
crossed = tbm.crossing_year(frames["baseline"]["prevalence"], threshold)
pd.DataFrame(
    {
        "baseline crosses threshold": crossed,
        "programme triggered": {
            tbm.SCENARIO_LABELS[n]: tbm.crossing_year(1.0 - f["programme_waiting"], 0.5)
            for n, f in frames.items()
            if n != "baseline"
        },
    }
).assign(difference_days=lambda d: 365 * (d["programme triggered"] - d["baseline crosses threshold"])).round(3)

**Does it match switching on by date?** The same consequence gives a second check. Running the
baseline, reading off the crossing year, and switching the programme on *at that date* must
give the same answer as the in-model trigger. The trigger signal is an argument to
`build_model`, so a date-triggered programme is the same model with `Time()` as its signal and
the crossing year as its threshold. The two prevalence curves should lie on top of each other.

So why trigger inside the model at all? Because the date is only known *after* a baseline run,
and it is different for every parameter set. Every draw from a calibration posterior, every
threshold in a sweep, and every sensitivity analysis would need its own two-pass run. A
reactive policy (section 6) cannot be done by date at all. The in-model trigger handles all of
these in a single run.

In [ ]:
by_date = tbm.build_model(trigger_signal=Time()).compile()
dated = tbm.indicators(
    tbm.run(by_date, tbm.scenario_params("acf_triggered", threshold=crossed, threshold_width=0.001))
)
compare = pd.DataFrame(
    {
        "triggered by prevalence (in the ODE)": frames["acf_triggered"]["prevalence"],
        f"switched on by date ({crossed:.2f})": dated["prevalence"],
        "baseline": frames["baseline"]["prevalence"],
    }
).loc[2020:2045]
compare.plot(
    title="Prevalence trigger vs switching on at the baseline crossing date",
    labels={"year": "year", "value": "prevalence per 100,000", "variable": ""},
)

## 6. The reactive policy is a thermostat

`acf_reactive` sets `stand_down` to 365 per year. The programme winds down within days whenever
prevalence drops below 350 and restarts whenever it rises above. In a deterministic model this
behaves like a thermostat. Prevalence is held at the threshold, and the programme runs at
whatever partial intensity is needed to keep it there.

The figure shows that intensity. It peaks during the transmission rise and then eases as the
epidemic's momentum falls. The policy never lets prevalence fall below 350, which is why it
averts the least of the four.

In reality a reactive programme responds to surveys and reviews with delays, which would
produce cycles rather than a flat line. Modelling that would mean adding the delay as more
compartments in the programme side model.

In [ ]:
reactive = frames["acf_reactive"].loc[2025:2050]
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=reactive.index, y=reactive["prevalence"], name="prevalence per 100,000")
fig.add_scatter(x=reactive.index, y=reactive["programme_active"], name="programme coverage", line_dash="dot", secondary_y=True)
fig.add_hline(y=threshold, line_dash="dash")
fig.update_layout(title="Reactive ACF: prevalence held at the threshold", xaxis_title="year")
fig.update_yaxes(title_text="prevalence per 100,000", secondary_y=False)
fig.update_yaxes(title_text="coverage", range=[0, 1], secondary_y=True)
fig

## 7. A modelling pitfall: deciding more slowly than the intervention acts

The latch works because the *decision* (`trigger`, mean about one day) is much faster than the
*effect* (roll-out, then ACF depleting prevalent cases). If deciding is slow, the first part of
the programme can push prevalence back below the threshold before the rest of the mass has
left `waiting`. The remaining mass then stays in `waiting` for good, and the programme stalls
at partial coverage.

The figure reruns `acf_triggered` with `trigger_speed` of 365, 5 and 1 per year. Only the
fastest reaches full coverage. If your policy means "once decided, fully committed", keep
`trigger_speed` large. Model the delays you care about with `rollout_years`, or with more
compartments between `scaling_up` and `active`, rather than by slowing the decision.

In [ ]:
speeds = {
    f"trigger_speed {s:g}/yr": tbm.indicators(
        tbm.run(cm, tbm.scenario_params("acf_triggered", trigger_speed=s))
    )["programme_active"]
    for s in (365.0, 5.0, 1.0)
}
pd.DataFrame(speeds).loc[2025:2045].plot(
    title="Coverage reached when the decision is slow",
    labels={"year": "year", "value": "programme coverage", "variable": ""},
)

## Try it

Everything is a parameter, so the scenarios take one line each to change:

```python
frame = tbm.indicators(tbm.run(cm, tbm.scenario_params("acf_triggered", threshold=300.0, rollout_years=1.0)))
```

* `threshold`: earlier or later trigger. Above about 475 the baseline never reaches it.
* `rollout_years`: speed of scale-up.
* `acf_screening`, `acf_sensitivity`, `tpt_rate`, `tpt_efficacy`: intervention strength.
* `stand_down`: anything above zero turns a latched programme into a reactive one.

Next: [`03-thresholds-and-signals`](03-thresholds-and-signals.ipynb) sweeps the threshold in
one vectorised run and triggers on notifications instead of prevalence.